In [55]:
from datasets import  load_dataset

In [56]:
dataset = load_dataset("fancyzhx/ag_news")


In [57]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [58]:
dataset["train"][500]

{'text': "Kerry Campaign Helping With Fla. Recovery (AP) AP - Democratic presidential candidate John Kerry does not plan to visit Florida in the aftermath of Hurricane Charley because he's concerned his campaign entourage could distract from recovery efforts, he said Saturday.",
 'label': 0}

In [59]:
label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

In [60]:
dataset["train"][0]["label"]

2

In [61]:
label_names[dataset["train"][0]["label"]]

'Business'

In [62]:
for i in range(100,105):
    print("TEXT:", dataset["train"][i]["text"])
    print("LABEL:", label_names[dataset["train"][i]["label"]])
    print("-" * 80)

TEXT: Comets, Asteroids and Planets around a Nearby Star (SPACE.com) SPACE.com - A nearby star thought to harbor comets and asteroids now appears to be home to planets, too. The presumed worlds are smaller than Jupiter and could be as tiny as Pluto, new observations suggest.
LABEL: Sci/Tech
--------------------------------------------------------------------------------
TEXT: Perseid Meteor Shower Peaks Overnight (SPACE.com) SPACE.com - A fine display of shooting stars is underway and peaks overnight Wednesday into early Thursday morning. Astronomers expect the 2004 Perseid meteor shower to be one of the best versions of the annual event in several years.
LABEL: Sci/Tech
--------------------------------------------------------------------------------
TEXT: Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar system is nbsp;exciting, the first task for astronauts and robots alike is to actual

In [63]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
        "distilbert-base-uncased"
)


In [64]:
text =" Apple released a new phone "
tokens = tokenizer(text)
tokens

{'input_ids': [101, 6207, 2207, 1037, 2047, 3042, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [65]:
def tokenize_function (batch):
    return tokenizer(
        batch["text"],
        padding= "max_length",
        truncation =True,
        max_length =128
    )

tokenized_dataset =dataset.map(
    tokenize_function,
    batched=True
)


In [66]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["text"]
)

In [67]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [68]:
from transformers import AutoModelForSequenceClassification



In [69]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3055.27it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [70]:
import evaluate

accuracy = evaluate.load('accuracy')

In [71]:
def compute_metrics(eval_pred):
    predictions ,labels = eval_pred

    predictions = predictions.argmax(axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [72]:
from transformers import TrainingArguments

In [73]:


training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none",
    fp16=True,
    dataloader_num_workers=2
)

In [74]:
from transformers import Trainer


In [75]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

In [76]:
trainer.train()

Step,Training Loss
500,0.426465
1000,0.265597
1500,0.239098
2000,0.228348
2500,0.230571
3000,0.244032
3500,0.208728
4000,0.212990
4500,0.202684
5000,0.196601


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.80it/s]


TrainOutput(global_step=15000, training_loss=0.18276282882690428, metrics={'train_runtime': 1846.018, 'train_samples_per_second': 130.01, 'train_steps_per_second': 8.126, 'total_flos': 7948327403520000.0, 'train_loss': 0.18276282882690428, 'epoch': 2.0})

In [77]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Step,Accuracy
0.137138,0.187013,15000,0.948947


{'eval_loss': 0.18701297044754028, 'eval_accuracy': 0.9489473684210527}

In [78]:
trainer.save_model("./news_classifier")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.50it/s]


In [79]:
tokenizer.save_pretrained("./news_classifier")

('./news_classifier\\tokenizer_config.json',
 './news_classifier\\tokenizer.json')

In [80]:
import torch

In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():

        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0][predicted_class].item()

    return predicted_class, confidence

In [82]:
text = "The company announced a new artificial intelligence chip."

predicted_class, confidence = predict(text)

print("Category:", label_names[predicted_class])
print("Confidence:", confidence)

Category: Sci/Tech
Confidence: 0.981561005115509


In [83]:
examples = [
    "The football team won the championship after scoring two goals.",
    "The company reported record profits this quarter.",
    "Scientists developed a new artificial intelligence system.",
    "The president met with international leaders to discuss the conflict."
]

for text in examples:
    category, confidence = predict(text)
    print("\nText:", text)
    print("Category:", category)
    print("Confidence:", f"{confidence:.2%}")


Text: The football team won the championship after scoring two goals.
Category: 1
Confidence: 49.61%

Text: The company reported record profits this quarter.
Category: 2
Confidence: 98.04%

Text: Scientists developed a new artificial intelligence system.
Category: 3
Confidence: 98.04%

Text: The president met with international leaders to discuss the conflict.
Category: 0
Confidence: 99.59%
